# 05 — Exploration des coûts de transaction (fork S3)

**Objectif.** Calibrer le fork coûts (README §8bis) sur de **vrais quotes** `opprcd` : spread
bid-ask relatif des options ~ATM, maturité 60–120 j, pour **SPX** vs **méga-caps** (rnk 1–10)
vs **petites lignes du top-100** (rnk 90–100), à 5 époques (compression des frictions attendue :
décimalisation 2001, penny pilot 2007, market-making électronique).

**Modèle de coût visé** (convention DMV « spread payé une fois », held-to-maturity) :
coût d'entrée par jambe = ½ × spread relatif × prime (on traverse le demi-spread vs le mid,
`impl_premium` étant un prix mid-like) ; + coût du delta-hedge quotidien (~1 bp × |Δn|×S).
Règlement à l'échéance = pas de spread (exercice/cash settlement).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from dispersion.data.wrds_client import get_connection

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
db = get_connection()
tabs = db.raw_sql("""SELECT table_name FROM information_schema.tables
                     WHERE table_schema='optionm' AND table_name LIKE 'opprcd%%' LIMIT 3""")
print(tabs.to_string())

Loading library list...


Done
   table_name
0  opprcd1996
1  opprcd1997
2  opprcd1998


In [2]:
# Échantillonnage robustifié : 3 dates/époque poolées, OI > 0, options ~ATM 60-120 j
w = pd.read_parquet(ROOT / "data" / "processed" / "weights.parquet")
w["rebalance_date"] = pd.to_datetime(w["rebalance_date"])
eras = {"1998": ["1998-03-31", "1998-06-30", "1998-09-30"],
        "2005": ["2005-03-31", "2005-06-30", "2005-09-30"],
        "2012": ["2012-03-30", "2012-06-29", "2012-09-28"],
        "2020": ["2020-03-31", "2020-06-30", "2020-09-30"],
        "2024": ["2024-03-28", "2024-06-28", "2024-09-30"]}

rows = []
for year, dates in eras.items():
    wq = w[w["rebalance_date"] == dates[1]].dropna(subset=["secid"])   # univers de mi-année
    groups = {
        "SPX": [108105],
        "mega (rnk 1-10)": wq[wq["rnk"] <= 10]["secid"].astype(int).tolist(),
        "small (rnk 90-100)": wq[wq["rnk"] >= 90]["secid"].astype(int).tolist(),
    }
    date_list = ", ".join(f"'{d}'" for d in dates)
    for label, secs in groups.items():
        q = f"""
        SELECT secid, date, cp_flag, best_bid, best_offer
        FROM optionm.opprcd{year}
        WHERE secid IN ({', '.join(map(str, secs))})
          AND date IN ({date_list})
          AND (exdate - date) BETWEEN 60 AND 120
          AND ABS(delta) BETWEEN 0.35 AND 0.65
          AND best_bid > 0.05 AND best_offer > best_bid
          AND open_interest > 0
        """
        df = db.raw_sql(q)
        if df.empty:
            rows.append((year, label, np.nan, np.nan, np.nan, 0))
            continue
        rel = (df["best_offer"] - df["best_bid"]) / ((df["best_offer"] + df["best_bid"]) / 2)
        rows.append((year, label, float(rel.median()),
                     float(rel.quantile(0.25)), float(rel.quantile(0.75)), len(df)))

cal = pd.DataFrame(rows, columns=["era", "groupe", "median", "q25", "q75", "n_quotes"])
piv = cal.pivot(index="era", columns="groupe", values="median")
print("SPREAD RELATIF MÉDIAN (% de la prime, bid→ask complet) :")
print((piv * 100).round(1).to_string())
print()
cal.round(4)

SPREAD RELATIF MÉDIAN (% de la prime, bid→ask complet) :
groupe  SPX  mega (rnk 1-10)  small (rnk 90-100)
era                                             
1998    0.9              6.8                 9.8
2005    1.6              6.1                 5.4
2012    7.6              1.1                 2.6
2020    0.7              3.0                 7.9
2024    0.6              1.2                 3.1



,era,groupe,median,q25,q75,n_quotes
0,1998,SPX,0.0091,0.0036,0.0167,62
1,1998,mega (rnk 1-10),0.0678,0.0541,0.0870,147
2,1998,small (rnk 90-100),0.0984,0.0774,0.1224,103
3,2005,SPX,0.0161,0.0130,0.0192,50
4,2005,mega (rnk 1-10),0.0606,0.0408,0.0870,49
5,2005,small (rnk 90-100),0.0538,0.0408,0.0731,84
6,2012,SPX,0.0760,0.0675,0.0867,130
7,2012,mega (rnk 1-10),0.0106,0.0072,0.0169,334
8,2012,small (rnk 90-100),0.0260,0.0185,0.0444,145
9,2020,SPX,0.0071,0.0055,0.0118,1669


## Constats & calibration proposée

1. Hiérarchie attendue : SPX ≪ méga-caps < petites lignes ; compression au fil des époques.
2. Le coût d'entrée par jambe = **½ × spread relatif** (on paie le demi-spread vs mid).
3. → Grille paramétrique par époque et groupe, appliquée aux primes d'entrée dans l'engine
   (+ 1 bp sur le notionnel du hedge quotidien). Sensibilité ±50 % en robustesse S5.